# Gender and State Analysis of Nigeria MPI Indicators

This notebook examines selected multidimensional poverty indicators by sex and state using the `Individual_Weight` variable in the supplied gender-analysis dataset.

## 1. Setup and input path

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RAW_DIR = Path("../data/raw")
INPUT_FILE = RAW_DIR / "Gender_MPI.xlsx"


## 2. Load the data

In [ ]:
df = pd.read_excel(INPUT_FILE)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
df.info()

In [ ]:
df[["Nutrition", "Individual_Weight", "Population_Weight", "Household_Weight"]].head()

## 3. Weighted indicator prevalence by sex and state

For each indicator, weighted prevalence is calculated as the weighted sum of the indicator divided by the total individual weight within each sex-state group. The result is expressed as a percentage.

In [ ]:
indicator_columns = [
    "Nutrition", "Food_Security", "Health", "School_Attendance",
    "Years_of_Schooling", "School_Lag", "Housing", "Cooking",
    "Asset", "Unemployment", "underemployment", "Security_Shock"
]

required = ["Sex", "State", "Individual_Weight"] + indicator_columns
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

weighted_rows = df.copy()
for col in indicator_columns:
    weighted_rows[f"weighted_{col}"] = weighted_rows[col] * weighted_rows["Individual_Weight"]

summary = weighted_rows.groupby(["Sex", "State"]).agg(
    total_individual_weight=("Individual_Weight", "sum"),
    **{f"weighted_{col}": (f"weighted_{col}", "sum") for col in indicator_columns}
).reset_index()

for col in indicator_columns:
    summary[f"{col}_prevalence"] = (
        summary[f"weighted_{col}"] / summary["total_individual_weight"] * 100
    )

summary.head()

## 4. Nutrition by sex and state

In [ ]:
plot_df = summary.copy()

plt.figure(figsize=(14, 6))
sns.lineplot(
    data=plot_df,
    x="State",
    y="Nutrition_prevalence",
    hue="Sex",
    marker="o"
)
plt.xlabel("State")
plt.ylabel("Weighted nutrition deprivation (%)")
plt.title("Nutrition Deprivation by Sex and State")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=plot_df, x="Sex", y="Nutrition_prevalence", errorbar=None)
plt.xlabel("Sex")
plt.ylabel("Weighted nutrition deprivation (%)")
plt.title("Nutrition Deprivation by Sex")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.stripplot(
    data=plot_df,
    x="Sex",
    y="Nutrition_prevalence",
    jitter=True
)
plt.xlabel("Sex")
plt.ylabel("Weighted nutrition deprivation (%)")
plt.title("State-Level Nutrition Deprivation by Sex")
plt.tight_layout()
plt.show()

## 5. Export the summary

The summary table contains the weighted indicator totals and prevalence estimates by sex and state.

In [ ]:
OUTPUT_FILE = Path("../outputs/tables/gender_state_mpi_summary.csv")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")